In [44]:
import re
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from tqdm.auto import trange
import time

from corus import load_lenta

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

import matplotlib.pyplot as plt
import seaborn as sns

from navec import Navec

In [4]:
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [5]:
DATA_PATH = Path("lenta-ru-news.csv.gz")
print(DATA_PATH)

if not DATA_PATH.exists():
    !wget -O lenta-ru-news.csv.gz https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

assert DATA_PATH.exists(), "Не удалось скачать lenta-ru-news.csv.gz"
DATA_PATH

lenta-ru-news.csv.gz


PosixPath('lenta-ru-news.csv.gz')

In [6]:
rows = []

for rec in tqdm(load_lenta(str(DATA_PATH)), desc="Loading lenta"):
    rows.append({
        "title": rec.title,
        "text": rec.text,
        "topic": rec.topic
    })

df = pd.DataFrame(rows)
df.head()

Loading lenta: 0it [00:00, ?it/s]

,title,text,topic
0,Названы регионы России с самой высокой смертно...,Вице-премьер по социальным вопросам Татьяна Го...,Россия
1,Австрия не представила доказательств вины росс...,Австрийские правоохранительные органы не предс...,Спорт
2,Обнаружено самое счастливое место на планете,Сотрудники социальной сети Instagram проанализ...,Путешествия
3,В США раскрыли сумму расходов на расследование...,С начала расследования российского вмешательст...,Мир
4,Хакеры рассказали о планах Великобритании зами...,Хакерская группировка Anonymous опубликовала н...,Мир


In [7]:
print(df.shape)
df.info()

(739351, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 739351 entries, 0 to 739350
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   title   739351 non-null  object
 1   text    739351 non-null  object
 2   topic   739351 non-null  object
dtypes: object(3)
memory usage: 16.9+ MB


In [8]:
# Убираем пустые и пробельные темы
df["topic"] = df["topic"].astype(str).str.strip()
df = df[df["topic"] != ""].copy()

print(df.shape)
print(df["topic"].nunique())

(739148, 3)
23


In [9]:
MIN_CLASS_SIZE = 100
# убираю слишком редкие классы, чтобы стратификация и оценка были стабильнее

topic_counts = df["topic"].value_counts()
valid_topics = topic_counts[topic_counts >= MIN_CLASS_SIZE].index
df = df[df["topic"].isin(valid_topics)].copy()

print(df.shape)
print(df["topic"].nunique())

(739076, 3)
18


In [58]:
N_SAMPLES = 100_000

if len(df) > N_SAMPLES:
    df_sample, _ = train_test_split(
        df,
        train_size=N_SAMPLES,
        stratify=df["topic"],
        random_state=RANDOM_STATE
    )
else:
    df_sample = df.copy()

df_sample = df_sample.reset_index(drop=True)

print(df_sample.shape)
print((df_sample["topic"].value_counts(normalize=True) * 100).round(2).head(10))

(100000, 3)
topic
Россия             21.72
Мир                18.49
Экономика          10.76
Спорт               8.72
Культура            7.28
Бывший СССР         7.22
Наука и техника     7.19
Интернет и СМИ      6.04
Из жизни            3.74
Дом                 2.94
Name: proportion, dtype: float64


In [59]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
NON_WORD_RE = re.compile(r"[^0-9a-zа-яё\- ]+", flags=re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")

# не делаю стемминг/лемматизацию для word2vec, чтобы не терять различия словоформ и не усложнять пайплайн

def normalize_text(title: str, text: str) -> str:
    s = f"{title} {text}".lower()
    s = s.replace("\xa0", " ")
    s = URL_RE.sub(" ", s)
    s = NON_WORD_RE.sub(" ", s)
    s = MULTISPACE_RE.sub(" ", s).strip()
    return s

tqdm.pandas()

df_sample["clean_text"] = df_sample.progress_apply(
    lambda row: normalize_text(row["title"], row["text"]),
    axis=1
)

df_sample = df_sample[df_sample["clean_text"] != ""].reset_index(drop=True)
df_sample[["title", "topic", "clean_text"]].head()

  0%|          | 0/100000 [00:00<?, ?it/s]

,title,topic,clean_text
0,Подросток стрелял по людям из пневматического ...,Россия,подросток стрелял по людям из пневматического ...
1,Япония запустила самую маленькую ракету-носитель,Наука и техника,япония запустила самую маленькую ракету-носите...
2,Основатель «Пятерочки» продаст отель-усадьбу в...,Дом,основатель пятерочки продаст отель-усадьбу в к...
3,"На Украине опубликуют ""черный список антисемитов""",Бывший СССР,на украине опубликуют черный список антисемито...
4,Россия приступила к разработке аппарата для ис...,Наука и техника,россия приступила к разработке аппарата для ис...


In [60]:
def tokenize(text: str):
    return text.split()

df_sample["tokens"] = df_sample["clean_text"].progress_apply(tokenize)
df_sample = df_sample[df_sample["tokens"].map(len) > 0].reset_index(drop=True)

df_sample[["clean_text", "tokens"]].head()

  0%|          | 0/100000 [00:00<?, ?it/s]

,clean_text,tokens
0,подросток стрелял по людям из пневматического ...,"[подросток, стрелял, по, людям, из, пневматиче..."
1,япония запустила самую маленькую ракету-носите...,"[япония, запустила, самую, маленькую, ракету-н..."
2,основатель пятерочки продаст отель-усадьбу в к...,"[основатель, пятерочки, продаст, отель-усадьбу..."
3,на украине опубликуют черный список антисемито...,"[на, украине, опубликуют, черный, список, анти..."
4,россия приступила к разработке аппарата для ис...,"[россия, приступила, к, разработке, аппарата, ..."


In [61]:
# df_sample = df_sample[:1000]

In [62]:
train_df, temp_df = train_test_split(
    df_sample,
    test_size=0.4,
    stratify=df_sample["topic"],
    random_state=RANDOM_STATE
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["topic"],
    random_state=RANDOM_STATE
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(train_df.shape, valid_df.shape, test_df.shape)

(60000, 5) (20000, 5) (20000, 5)


In [63]:
X_train_text = train_df["clean_text"].tolist()
X_valid_text = valid_df["clean_text"].tolist()
X_test_text  = test_df["clean_text"].tolist()

X_train_tokens = train_df["tokens"].tolist()
X_valid_tokens = valid_df["tokens"].tolist()
X_test_tokens  = test_df["tokens"].tolist()

y_train = train_df["topic"].values
y_valid = valid_df["topic"].values
y_test  = test_df["topic"].values

## Word2Vec

In [64]:
w2v_model = Word2Vec(
    vector_size=200,
    window=5, # компромисс между локальным и более широким контекстом
    min_count=5, # отсекаем шумные токены
    workers=4,
    sg=1, # skip-gram лучше работает с редкими словами
    negative=10,
    seed=RANDOM_STATE
)

print("словарь...")
t0 = time.time()
w2v_model.build_vocab(X_train_tokens)
print(f"словарь построен за {time.time() - t0} сек")
print(f"Размер словаря: {w2v_model.corpus_total_words:,} токенов, {len(w2v_model.wv):,} уникальных слов")

print("обучение...")
for epoch in trange(10, desc="Training w2v"):
    t0 = time.time()
    w2v_model.train(
        X_train_tokens,
        total_examples=w2v_model.corpus_count,
        epochs=1
    )
    print(f"Эпоха {epoch + 1} завершена за {time.time() - t0} сек")

w2v_kv = w2v_model.wv
print(f"Размерность эмбеддингов: {w2v_kv.vector_size}")

словарь...
словарь построен за 4.101820707321167 сек
Размер словаря: 11,275,994 токенов, 108,227 уникальных слов
обучение...


Training w2v:   0%|          | 0/10 [00:00<?, ?it/s]

Эпоха 1 завершена за 47.045875787734985 сек
Эпоха 2 завершена за 46.24284100532532 сек
Эпоха 3 завершена за 46.56093788146973 сек
Эпоха 4 завершена за 46.744221210479736 сек
Эпоха 5 завершена за 46.28986692428589 сек
Эпоха 6 завершена за 46.34423756599426 сек
Эпоха 7 завершена за 47.247162103652954 сек
Эпоха 8 завершена за 45.938313245773315 сек
Эпоха 9 завершена за 47.612446308135986 сек
Эпоха 10 завершена за 46.00763130187988 сек
Размерность эмбеддингов: 200


In [65]:
print(w2v_kv.index_to_key[:500])

['в', 'на', 'и', 'по', 'что', 'с', 'не', 'из', 'о', 'года', 'за', 'как', 'к', '-', 'для', 'его', 'а', 'этом', 'он', 'от', 'об', 'также', 'сообщает', 'россии', 'до', 'году', 'был', 'после', 'будет', 'время', 'это', 'при', 'у', 'сша', 'со', 'были', 'во', 'того', 'заявил', 'словам', 'долларов', 'было', 'однако', '1', 'ранее', 'их', 'том', 'который', 'тысяч', 'человек', 'уже', 'которые', 'но', 'более', 'они', 'ее', 'еще', 'рублей', 'около', 'все', 'данным', 'компании', 'процентов', 'под', 'так', 'была', '2', 'лет', '5', 'этого', 'то', 'будут', 'процента', 'же', 'может', 'миллионов', 'она', 'только', 'кроме', 'чтобы', 'страны', '3', 'новости', 'или', 'является', 'из-за', 'несколько', 'риа', 'один', 'тем', 'суд', 'рф', 'президента', 'пока', '4', 'власти', 'the', 'результате', 'между', 'президент', 'украины', 'два', '10', '6', 'против', 'мы', 'сообщил', 'когда', 'если', 'решение', 'области', 'ссылкой', 'глава', 'которая', 'всего', 'компания', 'где', 'я', '20', 'которых', 'них', '7', 'отметил'

In [66]:
probe_words = ["нефть", "российские", "рубля"]

for word in probe_words:
    if word in w2v_kv:
        print(f"\nmost_similar('{word}'):")
        print(w2v_kv.most_similar(word, topn=10))


most_similar('нефть'):
[('баррель', 0.7453940510749817), ('brent', 0.7320024967193604), ('барреля', 0.7158852815628052), ('нефти', 0.710671603679657), ('urals', 0.7054430246353149), ('североморскую', 0.6957404613494873), ('августовские', 0.6952854990959167), ('фьючерсов', 0.6924294233322144), ('фьючерсы', 0.6920465230941772), ('сырую', 0.6911002397537231)]

most_similar('российские'):
[('американские', 0.6857210993766785), ('отечественные', 0.6403515934944153), ('украинские', 0.6258518695831299), ('румынские', 0.6246842741966248), ('иностранные', 0.6242607235908508), ('казахстанские', 0.621875524520874), ('французские', 0.6100724339485168), ('норвежские', 0.6096036434173584), ('финские', 0.6043763160705566), ('европейские', 0.5916397571563721)]

most_similar('рубля'):
[('доллара', 0.8204084634780884), ('копеек', 0.8022236824035645), ('копейку', 0.778289794921875), ('копейки', 0.7685164213180542), ('доллар', 0.7628366947174072), ('бивалютная', 0.7212367653846741), ('рубль', 0.716838121

In [67]:
triplets = [
    ["россия", "сша", "германии", "рубля"],
    ["банк", "нефть", "долларов", "самолет"],
    ["путин", "президент", "министр", "самолет"],
]

for words in triplets:
    existing = [w for w in words if w in w2v_kv]
    print(existing, "->", w2v_kv.doesnt_match(existing))

['россия', 'сша', 'германии', 'рубля'] -> рубля
['банк', 'нефть', 'долларов', 'самолет'] -> самолет
['путин', 'президент', 'министр', 'самолет'] -> самолет


intrinsic оценка эмбеддингов показала, что модель улавливает тематическую близость слов.
В целом качество обученных эмбеддингов можно считать приемлемым

## Navec

In [46]:
NAVEC_PATH = Path("navec_news_v1_1B_250K_300d_100q.tar")

if not NAVEC_PATH.exists():
    !wget -O {NAVEC_PATH} https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

assert NAVEC_PATH.exists(), "Не удалось скачать модель Navec"

--2026-03-14 16:45:43--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving mlcore-webproxy.mlcore-webproxy.svc.cluster.local (mlcore-webproxy.mlcore-webproxy.svc.cluster.local)... 10.32.0.166
Connecting to mlcore-webproxy.mlcore-webproxy.svc.cluster.local (mlcore-webproxy.mlcore-webproxy.svc.cluster.local)|10.32.0.166|:3128... connected.
Proxy request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  77.2MB/s    in 0.3s    

2026-03-14 16:45:44 (77.2 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [47]:
navec = Navec.load(str(NAVEC_PATH))
print("Navec loaded")
print("Размерность:", navec.pq.dim)

Navec loaded
Размерность: 300


## rus-vectores

In [53]:
import urllib.request
import gensim

urllib.request.urlretrieve(
    "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
)

model_ru = gensim.models.KeyedVectors.load_word2vec_format(
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    binary=False
)
model_ru.most_similar(positive=["ночь_NOUN"], topn=10)

[('ночь_PROPN', 0.7704508304595947),
 ('вечер_NOUN', 0.7683228254318237),
 ('утро_NOUN', 0.7520124316215515),
 ('полночь_NOUN', 0.7201331853866577),
 ('рассвет_NOUN', 0.6792924404144287),
 ('полдень_NOUN', 0.6637035012245178),
 ('утро_PROPN', 0.6531521677970886),
 ('ночь_ADV', 0.6248846650123596),
 ('сумерки_NOUN', 0.6227153539657593),
 ('ночью_NOUN', 0.6219336986541748)]

In [55]:
rusvectores_kv = model_ru

print("RusVectores loaded")
print("Размер словаря:", len(rusvectores_kv))
print("Размерность:", rusvectores_kv.vector_size)

RusVectores loaded
Размер словаря: 384764
Размерность: 300


In [68]:
def get_w2v_vector(token: str):
    if token in w2v_kv:
        return w2v_kv[token]
    return None

NAVEC_DIM = navec.pq.dim

def get_navec_vector(token: str):
    if token in navec:
        return navec[token]
    return None

RUSVEC_DIM = rusvectores_kv.vector_size

def get_rusvectores_vector(token: str):
    candidates = [
        token,
        f"{token}_NOUN",
        f"{token}_PROPN",
        f"{token}_ADJ",
        f"{token}_VERB",
        f"{token}_ADV",
    ]
    for cand in candidates:
        if cand in rusvectores_kv:
            return rusvectores_kv[cand]
    return None

In [70]:
# покрытие словаря

def embedding_coverage(tokens_list, get_vector_fn):
    total = 0
    found = 0
    for tokens in tokens_list:
        for token in tokens:
            total += 1
            if get_vector_fn(token) is not None:
                found += 1
    return found / total if total > 0 else 0.0


print(embedding_coverage(X_train_tokens, get_w2v_vector))
print(embedding_coverage(X_train_tokens, get_navec_vector))
print(embedding_coverage(X_train_tokens, get_rusvectores_vector))

0.9622206255164734
0.9406881557404163
0.5604244734433168


Покрытие словаря оказалось максимальным у собственного w2v и высоким у navec, тогда как у rus-vectores оно заметно ниже. Это ожидаемо, поскольку rus-vectores использует ключи формата lemma_POS, а в текущем пайплайне применялось лишь приближённое сопоставление токенов с несколькими возможными частями речи без полноценной лемматизации

In [71]:
def mean_pooling_vectors(tokens_list, get_vector_fn, dim):
    X = np.zeros((len(tokens_list), dim), dtype=np.float32)

    for i, tokens in enumerate(tqdm(tokens_list, desc="Vectorizing")):
        vectors = []
        for token in tokens:
            vec = get_vector_fn(token)
            if vec is not None:
                vectors.append(vec)

        if vectors:
            X[i] = np.mean(vectors, axis=0)

    return X

In [72]:
X_train_w2v = mean_pooling_vectors(X_train_tokens, get_w2v_vector, w2v_kv.vector_size)
X_valid_w2v = mean_pooling_vectors(X_valid_tokens, get_w2v_vector, w2v_kv.vector_size)
X_test_w2v  = mean_pooling_vectors(X_test_tokens,  get_w2v_vector, w2v_kv.vector_size)

Vectorizing:   0%|          | 0/60000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

In [73]:
X_train_navec = mean_pooling_vectors(X_train_tokens, get_navec_vector, NAVEC_DIM)
X_valid_navec = mean_pooling_vectors(X_valid_tokens, get_navec_vector, NAVEC_DIM)
X_test_navec  = mean_pooling_vectors(X_test_tokens,  get_navec_vector, NAVEC_DIM)

Vectorizing:   0%|          | 0/60000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

In [74]:
X_train_rusvec = mean_pooling_vectors(X_train_tokens, get_rusvectores_vector, RUSVEC_DIM)
X_valid_rusvec = mean_pooling_vectors(X_valid_tokens, get_rusvectores_vector, RUSVEC_DIM)
X_test_rusvec  = mean_pooling_vectors(X_test_tokens,  get_rusvectores_vector, RUSVEC_DIM)

Vectorizing:   0%|          | 0/60000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

Vectorizing:   0%|          | 0/20000 [00:00<?, ?it/s]

In [75]:
def fit_eval_logreg(X_train, y_train, X_valid, y_valid, name):
    clf = LogisticRegression(
        max_iter=500,
        solver="saga",
        verbose=1,
        multi_class="multinomial",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
    clf.fit(X_train, y_train)

    valid_pred = clf.predict(X_valid)

    return {
        "model": name,
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "valid_macro_f1": f1_score(y_valid, valid_pred, average="macro"),
        "clf": clf}

results = []

results.append(fit_eval_logreg(X_train_w2v, y_train, X_valid_w2v, y_valid, "Word2Vec mean"))
results.append(fit_eval_logreg(X_train_navec, y_train, X_valid_navec, y_valid, "Navec mean"))
results.append(fit_eval_logreg(X_train_rusvec, y_train, X_valid_rusvec, y_valid, "RusVectores mean"))

results_df = pd.DataFrame([
    {k: v for k, v in row.items() if k != "clf"}
    for row in results
]).sort_values("valid_macro_f1", ascending=False)

results_df

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.13289697
Epoch 3, change: 0.067943677
Epoch 4, change: 0.046195921
Epoch 5, change: 0.035057541
Epoch 6, change: 0.026700307
Epoch 7, change: 0.021355497
Epoch 8, change: 0.015639534
Epoch 9, change: 0.012343589
Epoch 10, change: 0.010679713
Epoch 11, change: 0.0086259888
Epoch 12, change: 0.0072071808
Epoch 13, change: 0.0055446355
Epoch 14, change: 0.0046034907
Epoch 15, change: 0.0039216792
Epoch 16, change: 0.0031741071
Epoch 17, change: 0.0027647919
Epoch 18, change: 0.0023275483
Epoch 19, change: 0.0021273778
Epoch 20, change: 0.0019924678
Epoch 21, change: 0.0018640643
Epoch 22, change: 0.0017640282
Epoch 23, change: 0.0016635741
Epoch 24, change: 0.0015796412
Epoch 25, change: 0.0015306076
Epoch 26, change: 0.0014594595
Epoch 27, change: 0.001330679
Epoch 28, change: 0.0012308072
Epoch 29, change: 0.0011760833
Epoch 30, change: 0.0011120215
Epoch 31, change: 0.0011028408
Epoch 32, change: 0.0010492377
Epoch 33, change: 0.001035742
Epoch 34,

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.1880431
Epoch 3, change: 0.10693125
Epoch 4, change: 0.069652401
Epoch 5, change: 0.051453803
Epoch 6, change: 0.036587723
Epoch 7, change: 0.028738484
Epoch 8, change: 0.028178325
Epoch 9, change: 0.018461769
Epoch 10, change: 0.016280364
Epoch 11, change: 0.013534416
Epoch 12, change: 0.0112827
Epoch 13, change: 0.0090978798
Epoch 14, change: 0.007966809
Epoch 15, change: 0.0068978518
Epoch 16, change: 0.005886869
Epoch 17, change: 0.0051383115
Epoch 18, change: 0.0046353019
Epoch 19, change: 0.004092813
Epoch 20, change: 0.0036071436
Epoch 21, change: 0.0031913924
Epoch 22, change: 0.0029221815
Epoch 23, change: 0.0027120861
Epoch 24, change: 0.0024989331
Epoch 25, change: 0.002283226
Epoch 26, change: 0.0021076133
Epoch 27, change: 0.0019638322
Epoch 28, change: 0.0017882579
Epoch 29, change: 0.0016494537
Epoch 30, change: 0.0015222876
Epoch 31, change: 0.001415165
Epoch 32, change: 0.0013111699
Epoch 33, change: 0.0012082049
Epoch 34, change: 

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.17750409
Epoch 3, change: 0.056719121
Epoch 4, change: 0.042610433
Epoch 5, change: 0.018368538
Epoch 6, change: 0.010843163
Epoch 7, change: 0.007619123
Epoch 8, change: 0.0053913351
Epoch 9, change: 0.00303673
Epoch 10, change: 0.002093629
Epoch 11, change: 0.001949748
Epoch 12, change: 0.0010909371
Epoch 13, change: 0.00052337721
Epoch 14, change: 0.00035195972
Epoch 15, change: 0.00019934599
Epoch 16, change: 0.00014286446
convergence after 17 epochs took 22 seconds


,model,valid_accuracy,valid_macro_f1
1,Navec mean,0.78940,0.557469
0,Word2Vec mean,0.77995,0.541244
2,RusVectores mean,0.71015,0.412856


In [76]:
best_row = max(results, key=lambda x: x["valid_macro_f1"])
best_name = best_row["model"]
print("Лучший набор эмбеддингов на val:", best_name)

Лучший набор эмбеддингов на val: Navec mean


In [77]:
# автовыбор для воспроизводимости пайплайна

if best_name == "Word2Vec mean":
    best_get_vector = get_w2v_vector
    BEST_DIM = w2v_kv.vector_size
elif best_name == "Navec mean":
    best_get_vector = get_navec_vector
    BEST_DIM = navec.pq.dim
else:
    best_get_vector = get_rusvectores_vector
    BEST_DIM = rusvectores_kv.vector_size

print(BEST_DIM)

300


## tfidf weighted avg

In [78]:
tfidf = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    min_df=5,
    max_df=0.95
)

tfidf.fit(X_train_tokens)
print(len(tfidf.vocabulary_))

96763


In [80]:
def tfidf_weighted_embeddings(tokens_list, tfidf_vectorizer, get_vector_fn, dim):
    X_tfidf = tfidf_vectorizer.transform(tokens_list)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    X = np.zeros((len(tokens_list), dim), dtype=np.float32)

    for i in tqdm(range(X_tfidf.shape[0]), desc="TF-IDF weighted embeddings"):
        row = X_tfidf[i]
        if row.nnz == 0:
            continue

        weighted_sum = np.zeros(dim, dtype=np.float32)
        weight_total = 0.0

        for idx, weight in zip(row.indices, row.data):
            token = feature_names[idx]
            vec = get_vector_fn(token)
            if vec is not None:
                weighted_sum += weight * vec
                weight_total += weight

        if weight_total > 0:
            X[i] = weighted_sum / weight_total

    return X


X_train_best_tfidf = tfidf_weighted_embeddings(X_train_tokens, tfidf, best_get_vector, BEST_DIM)
X_valid_best_tfidf = tfidf_weighted_embeddings(X_valid_tokens, tfidf, best_get_vector, BEST_DIM)
X_test_best_tfidf  = tfidf_weighted_embeddings(X_test_tokens,  tfidf, best_get_vector, BEST_DIM)

TF-IDF weighted embeddings:   0%|          | 0/60000 [00:00<?, ?it/s]

TF-IDF weighted embeddings:   0%|          | 0/20000 [00:00<?, ?it/s]

TF-IDF weighted embeddings:   0%|          | 0/20000 [00:00<?, ?it/s]

In [81]:
tfidf_emb_result = fit_eval_logreg(
    X_train_best_tfidf, y_train,
    X_valid_best_tfidf, y_valid,
    f"{best_name} + TF-IDF weighted"
)

pd.DataFrame([{
    k: v for k, v in tfidf_emb_result.items() if k != "clf"
}])

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.21552482
Epoch 3, change: 0.10587612
Epoch 4, change: 0.089359298
Epoch 5, change: 0.060005438
Epoch 6, change: 0.039998822
Epoch 7, change: 0.031096507
Epoch 8, change: 0.029871687
Epoch 9, change: 0.020063164
Epoch 10, change: 0.016720925
Epoch 11, change: 0.014630031
Epoch 12, change: 0.012971854
Epoch 13, change: 0.010958908
Epoch 14, change: 0.010155462
Epoch 15, change: 0.0086390469
Epoch 16, change: 0.0078104963
Epoch 17, change: 0.0068702735
Epoch 18, change: 0.0058081071
Epoch 19, change: 0.0054308721
Epoch 20, change: 0.0048467722
Epoch 21, change: 0.0043664421
Epoch 22, change: 0.0039866325
Epoch 23, change: 0.0035812533
Epoch 24, change: 0.0032464075
Epoch 25, change: 0.0030978401
Epoch 26, change: 0.0027105135
Epoch 27, change: 0.0025139095
Epoch 28, change: 0.0023397342
Epoch 29, change: 0.0022356326
Epoch 30, change: 0.0021473672
Epoch 31, change: 0.0021228374
Epoch 32, change: 0.0020398186
Epoch 33, change: 0.0019099284
Epoch 34, ch

,model,valid_accuracy,valid_macro_f1
0,Navec mean + TF-IDF weighted,0.78865,0.562871


In [82]:
trained_models = {row["model"]: row["clf"] for row in results}
trained_models[tfidf_emb_result["model"]] = tfidf_emb_result["clf"]

In [83]:
test_results = []

pred = trained_models["Word2Vec mean"].predict(X_test_w2v)
test_results.append({
    "model": "Word2Vec mean",
    "test_accuracy": accuracy_score(y_test, pred),
    "test_macro_f1": f1_score(y_test, pred, average="macro")
})

pred = trained_models["Navec mean"].predict(X_test_navec)
test_results.append({
    "model": "Navec mean",
    "test_accuracy": accuracy_score(y_test, pred),
    "test_macro_f1": f1_score(y_test, pred, average="macro")
})

pred = trained_models["RusVectores mean"].predict(X_test_rusvec)
test_results.append({
    "model": "RusVectores mean",
    "test_accuracy": accuracy_score(y_test, pred),
    "test_macro_f1": f1_score(y_test, pred, average="macro")
})

pred = trained_models[f"{best_name} + TF-IDF weighted"].predict(X_test_best_tfidf)
test_results.append({
    "model": f"{best_name} + TF-IDF weighted",
    "test_accuracy": accuracy_score(y_test, pred),
    "test_macro_f1": f1_score(y_test, pred, average="macro")
})

test_results_df = pd.DataFrame(test_results).sort_values("test_macro_f1", ascending=False)
test_results_df

,model,test_accuracy,test_macro_f1
3,Navec mean + TF-IDF weighted,0.78780,0.570545
1,Navec mean,0.78825,0.561815
0,Word2Vec mean,0.77745,0.536563
2,RusVectores mean,0.71320,0.416954


In [85]:
best_test_model_name = test_results_df.iloc[0]["model"]
print("Лучшая модель на test:", best_test_model_name)

Лучшая модель на test: Navec mean + TF-IDF weighted


In [86]:
if best_test_model_name == "Word2Vec mean":
    best_test_pred = trained_models[best_test_model_name].predict(X_test_w2v)
elif best_test_model_name == "Navec mean":
    best_test_pred = trained_models[best_test_model_name].predict(X_test_navec)
elif best_test_model_name == "RusVectores mean":
    best_test_pred = trained_models[best_test_model_name].predict(X_test_rusvec)
else:
    best_test_pred = trained_models[best_test_model_name].predict(X_test_best_tfidf)

print(classification_report(y_test, best_test_pred))

                   precision    recall  f1-score   support

   69-я параллель       0.82      0.26      0.40        34
           Бизнес       0.54      0.21      0.31       200
      Бывший СССР       0.81      0.77      0.79      1445
              Дом       0.82      0.76      0.79       588
         Из жизни       0.63      0.54      0.58       747
   Интернет и СМИ       0.74      0.67      0.70      1209
             Крым       0.00      0.00      0.00        18
     Культпросвет       0.00      0.00      0.00         9
         Культура       0.84      0.86      0.85      1456
          Легпром       0.00      0.00      0.00         3
              Мир       0.78      0.84      0.81      3699
  Наука и техника       0.79      0.83      0.81      1438
      Путешествия       0.64      0.47      0.54       174
           Россия       0.75      0.81      0.78      4344
Силовые структуры       0.45      0.24      0.31       531
            Спорт       0.96      0.96      0.96      1

## Вывод

Лучший результат на тестовой выборке показала модель **Navec mean + TF-IDF weighted**: `accuracy = 0.7878`, `macro_f1 = 0.5705`. Обычное усреднение эмбеддингов `Navec` также показало сильный результат (`accuracy = 0.7883`, `macro_f1 = 0.5618`), однако tf-idf-взвешивание позволило немного улучшить `macro_f1`, что особенно важно при несбалансированных классах.

Собственный `Word2Vec` показал результат ниже (`accuracy = 0.7775`, `macro_f1 = 0.5366`), но всё равно остался конкурентоспособным. `RusVectores` сработал заметно хуже (`accuracy = 0.7132`, `macro_f1 = 0.4170`), что согласуется с более низким покрытием словаря. Вероятная причина заключается в том, что `RusVectores` использует представление слов в формате `lemma_POS`, а у нас не применялась полноценная лемматизация и морфологическая разметка.

По `classification_report` видно, что модель хорошо распознаёт крупные и тематически устойчивые классы, такие как **«Спорт»**, **«Культура»**, **«Наука и техника»**, **«Экономика»**, **«Россия»** и **«Мир»**. Наиболее слабые результаты наблюдаются на редких классах, например **«Крым»**, **«Культпросвет»**, **«Легпром»** и **«69-я параллель»**, что ожидаемо из-за малого числа примеров и дисбаланса классов.